## 1. Imports & Dataset Loading

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from scipy.stats import randint, uniform

import xgboost as xgb

df = pd.read_csv('../dataset/predict_prices_dataset.csv')
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,A1,2017,99,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


## 2. Baseline — All Features, Native Categorical Support, No Tuning

In [2]:
categorical_cols = ['Make', 'fuelType', 'transmission', 'model']

for col in categorical_cols:
    df[col] = df[col].astype('category')

x = df.drop(columns=['price'])
y = df['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    enable_categorical=True,
    learning_rate=0.01,
    n_estimators=2000,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

model.fit(x_train, y_train)

y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       1465.4365   1918.8252
MAE         997.6738   1095.0702
R²            0.9750      0.9585


XGBoost also supports categorical columns natively, but it needs **two flags** working together: the pandas dtype set to `category` (same as LightGBM) **and** `enable_categorical=True` on the estimator. Without the second flag, categoricals raise an error rather than being silently ignored. `tree_method='hist'` selects histogram-based split finding — matches LightGBM's default and is what makes both libraries fast on larger datasets.

Structural difference to keep in mind: **XGBoost grows level-wise** (all leaves at the current depth are split before moving deeper), while LightGBM grows leaf-wise. That's why there's no `num_leaves` parameter here — tree shape is controlled by `max_depth` alone.

## Baseline Comparison

| Setup | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| LightGBM baseline (same features) | 0.9705 | 0.9562 | 1971 | 1142 | 0.014 |
| **XGBoost baseline (this cell)** | **0.9750** | **0.9585** | **1919** | **1095** | **0.017** |

XGBoost lands slightly ahead on test out of the box (~+0.25 pp R², -2.6% RMSE), while fitting train marginally harder (gap 0.017 vs 0.014). Very close — expected, since both are second-order gradient boosting on histograms with L1/L2 regularization. The default growth strategy is the main difference, and on data with strong categorical signal, XGBoost's balanced trees catch the pattern a hair sooner.

## 3. Full Pipeline — FE + Outlier Removal + Tuning with CV & Early Stopping

In [3]:
df['km_per_year'] = df['mileage'] / (2024 - df['year'] + 1)

df_clean = df[df['price'] >= 500]
df_clean = df_clean[df_clean['mpg'] <= 400]
df_clean = df_clean[df_clean['year'] >= 2001]

X = df_clean.drop(columns=['price'])
y = df_clean['price']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Val slice for early stopping (out of train, not test!)
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

base_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    enable_categorical=True,
    early_stopping_rounds=50,
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

param_distributions = {
    'n_estimators':      randint(500, 3000),
    'learning_rate':     uniform(0.005, 0.045),
    'max_depth':         randint(3, 8),
    'min_child_weight':  randint(1, 20),
    'gamma':             uniform(0.0, 1.0),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'reg_alpha':         uniform(0.1, 1.9),
    'reg_lambda':        uniform(0.1, 1.9),
}

fit_params = {
    'eval_set': [(x_val, y_val)],
    'verbose': False,
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

search.fit(x_tr, y_tr, **fit_params)

print("Cel mai bun scor CV (RMSE):", -search.best_score_)
print("Cei mai buni parametri:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

best_model = search.best_estimator_

# 7. Evaluate
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print()
print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Cel mai bun scor CV (RMSE): 1795.61572265625
Cei mai buni parametri:
  colsample_bytree: 0.7219125032632117
  gamma: 0.16465585314294173
  learning_rate: 0.02903402387189489
  max_depth: 7
  min_child_weight: 1
  n_estimators: 1757
  reg_alpha: 0.7414688256668931
  reg_lambda: 1.8915284374337864
  subsample: 0.7292811728083021

               Train        Test
RMSE       1266.0891   1708.7887
MAE         858.1071   1052.5958
R²            0.9814      0.9670


All the tuning apparatus from the LightGBM notebook rolled into a single cell:
- **Feature engineering**: `km_per_year = mileage / (2024 - year + 1)` added to `df`, then outlier trimming (`price ≥ 500`, `mpg ≤ 400`, `year ≥ 2001`).
- **Val slice for early stopping** carved from the train side (`x_train` → `x_tr` + `x_val`), so `x_test` is never touched during tuning.
- **`RandomizedSearchCV`** with 50 candidates × 5-fold CV = 250 fits, `neg_root_mean_squared_error` scoring.
- **Early stopping** via `early_stopping_rounds=50` + `eval_metric='rmse'` on the constructor (XGBoost's sklearn wrapper puts this on the estimator itself, not in a `callbacks` list like LightGBM).

Search space equivalent to LightGBM's constrained one, with two XGBoost-specific parameters:
- **`min_child_weight`** — minimum sum of hessian in a leaf; XGBoost's analog to LightGBM's `min_child_samples`, but computed on the second-order gradient rather than the raw row count.
- **`gamma`** — minimum loss reduction required to make a split; a structural regularizer LightGBM doesn't expose as prominently.

## Result

| Setup | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| XGBoost baseline (all features) | 0.9750 | 0.9585 | 1919 | 1095 | 0.017 |
| **XGBoost final (tuned + FE + outliers)** | **0.9814** | **0.9670** | **1709** | **1053** | **0.014** |

+0.85 pp on Test R², -11% on Test RMSE, gap slightly tightened. Same direction of gains as LightGBM.

## Head-to-Head with LightGBM

| Model | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| LightGBM (final) | 0.9784 | **0.9671** | 1708 | 1057 | 0.011 |
| XGBoost (final)  | 0.9814 | **0.9670** | 1709 | 1053 | 0.014 |

A photo finish — the two GBDT libraries land on the same test score to three decimal places (0.967 R², 1708–1709 RMSE, 1053–1057 MAE). XGBoost's train fit is slightly deeper (bigger gap), LightGBM's slightly shallower.

That's the expected outcome once feature engineering and hyperparameter tuning have converged: both models are the same underlying math (second-order gradient boosting with histogram split finding) with different growth strategies and different tuning ergonomics. On tabular data of this size, either one is a defensible choice.